# Tree model — **xgboost flavor** → CPU-SMALL serving endpoint

Trains an `XGBClassifier` and logs it with the native **`mlflow.xgboost`** flavor,
registers to Unity Catalog, and deploys a **CPU / SMALL** serving endpoint on the
shm-skunkworks FEVM. The served model's `/invocations` returns class labels.

Dependencies are pinned to the exact training-time versions.

In [ ]:
# Serverless job compute is bare — install what we need, then restart Python.
%pip install -q mlflow xgboost scikit-learn pandas numpy
dbutils.library.restartPython()

In [ ]:
CATALOG = "shm_catalog"
SCHEMA = "shared"
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.tree_xgboost_model"
ENDPOINT_NAME = "shm_tree_xgboost_endpoint"

In [ ]:
import mlflow, numpy as np, pandas as pd, sklearn, xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

mlflow.set_registry_uri("databricks-uc")

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.9, eval_metric="logloss", random_state=42,
)
clf.fit(X_train, y_train)
print("train acc:", clf.score(X_train, y_train), " test acc:", clf.score(X_test, y_test))

In [ ]:
from mlflow.models import infer_signature

example = X_test.head(3)
signature = infer_signature(example, clf.predict(example))

# Pin exact training-time versions so the serving container matches training.
pip_requirements = [
    f"mlflow=={mlflow.__version__}",
    f"xgboost=={xgb.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
]
print("pip_requirements:", pip_requirements)

with mlflow.start_run(run_name="tree_xgboost"):
    info = mlflow.xgboost.log_model(
        xgb_model=clf,
        artifact_path="model",
        signature=signature,
        input_example=example,
        registered_model_name=UC_MODEL_NAME,
        pip_requirements=pip_requirements,
    )
print("registered version:", info.registered_model_version)

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ServedModelInput,
    EndpointCoreConfigInput,
    ServedModelInputWorkloadSize,
    ServedModelInputWorkloadType,
)

w = WorkspaceClient()
served = ServedModelInput(
    model_name=UC_MODEL_NAME,
    model_version=info.registered_model_version,
    workload_type=ServedModelInputWorkloadType.CPU,     # CPU
    workload_size=ServedModelInputWorkloadSize.SMALL,   # SMALL
    scale_to_zero_enabled=True,
)

try:
    w.serving_endpoints.update_config(name=ENDPOINT_NAME, served_models=[served]).result()
    print(f"✅ updated endpoint {ENDPOINT_NAME}")
except Exception:
    w.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(served_models=[served]),
    ).result()
    print(f"✅ created endpoint {ENDPOINT_NAME}")

In [ ]:
from databricks.sdk.service.serving import DataframeSplitInput

resp = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_split=DataframeSplitInput(
        columns=list(X_test.columns),
        data=X_test.head(3).values.tolist(),
    ),
)
print("predictions:", resp.predictions)